# Merge velocyto loom files and subset to IL1R1 cells

Build a single merged loom from the per-sample velocyto outputs, then subset it to the cells present in the IL1R1 AnnData object, writing `selected.loom`.

## Setup

In [ ]:
import loompy
import numpy as np
from collections import defaultdict

## 1. Merge per-sample loom files into `merged.loom`

In [ ]:
files = ["/home/chanyue/Single_Cell/D3_correct/D3negative_RNA/velocyto/D3negative_RNA.loom",
         "/home/chanyue/Single_Cell/D3_correct/D3positive_RNA/velocyto/D3positive_RNA.loom",
         "/home/chanyue/Single_Cell/Prisila_raw_2/D4negative_good/D4negative_RNA/velocyto/D4negative_RNA.loom",
         "/home/chanyue/Single_Cell/Prisila_raw_2/D4positive_good/D4positive_RNA/velocyto/D4positive_RNA.loom",
         "/home/chanyue/Single_Cell/Prisila_raw_2/D5negative_good/D5negative_RNA/velocyto/D5negative_RNA.loom",
         "/home/chanyue/Single_Cell/Prisila_raw_2/D5positive_good/D5positive_RNA/velocyto/D5positive_RNA.loom",
         "/home/chanyue/Single_Cell/D6_and_D7/concatenated/D6negative_RNA/velocyto/D6negative_RNA.loom",
         "/home/chanyue/Single_Cell/D6_and_D7/concatenated/D6positive_RNA/velocyto/D6positive_RNA.loom",
         "/home/chanyue/Single_Cell/D6_and_D7/concatenated/D7negative_RNA/velocyto/D7negative_RNA.loom",
         "/home/chanyue/Single_Cell/D6_and_D7/concatenated/D7positive_RNA/velocyto/D7positive_RNA.loom"]

In [ ]:
cp /home/chanyue/Single_Cell/D3_correct/D3negative_RNA/velocyto/D3negative_RNA.loom merged.loom

In [ ]:
ds = loompy.connect("merged.loom")
for fn in files[1:]:
    ds.add_loom(fn, batch_size=1000)

## 2. Load IL1R1 AnnData and build donor-prefixed cell IDs

The loom `CellID`s are prefixed with the sample name (e.g. `D3negative_RNA:`). Re-create the same IDs for the IL1R1 cells so the two can be matched.

In [ ]:
import scanpy as sc

In [ ]:
IL1R1 = sc.read_h5ad("IL1R1_subsample.h5ad")

In [ ]:
IL1R1_cells = list(IL1R1.obs.index)

In [ ]:
Donors = ['D3negative_RNA:',
 'D3positive_RNA:',
 'D4negative_RNA:',
 'D4positive_RNA:',
 'D5negative_RNA:',
 'D5positive_RNA:',
 'D6negative_RNA:',
 'D6positive_RNA:',
 'D7negative_RNA:',
 'D7positive_RNA:']
Donors

In [ ]:
new_IL1R1_cells = []
for i in range(len(Donors)):
    for cell in IL1R1_cells:
        before, sep, after = cell.partition('-')
        if after == str(i + 1):
            new_IL1R1_cells.append(Donors[i] + cell)

### Sanity check: donor mapping matches loom samples

In [ ]:
index = [int(x) - 1 for x in np.unique(np.array([x.split("-")[1] for x in IL1R1_cells]))]

In [ ]:
temp = defaultdict(list)
for x in new_IL1R1_cells:
    before, sep, after = x.partition(':')
    temp[before+sep].append(after)

In [ ]:
set(np.array(Donors)[index]) == set(temp.keys())

## 3. Strip the 10x suffix to match the loom `CellID` format

In [ ]:
new_IL1R1_cells = [x.split("-")[0] + "x" for x in new_IL1R1_cells]

## 4. Subset `merged.loom` to the selected cells -> `selected.loom`

In [ ]:
ds = loompy.connect("merged.loom")

In [ ]:
boolvec = np.array([elem in new_IL1R1_cells for elem in ds.ca.CellID])

In [ ]:
with loompy.connect("selected.loom") as new_file:
    view = new_file.view[:, boolvec]

In [ ]:
ds.close()